# 第八课｜我们怎么知道硬件是对的？

上一课 RTL 看起来合理，但“看起来”不是验证。今天只解决：
> **没有真实 FPGA 板时，怎样给 RTL 输入、观察输出，并让错误自动失败？**

主要新概念：**硬件仿真（simulation）与测试平台（testbench）**。


## 1. 术语

**simulation**：软件执行 HDL 的逻辑/时序语义；不是实体芯片。

**testbench**：验证用 HDL，产生 clock/reset/input 并检查输出。

**waveform**：信号随仿真时间的变化图。

被测模块常简称 **DUT（Device Under Test）**。


## 2. testbench 不会进入最终硬件

testbench 可以用 `#5`、打印、`$fatal`、结束仿真；这些是验证行为。design RTL 才是未来综合成硬件的部分。


## 3. 先写 oracle，再看 waveform

波形不能靠“形状差不多”判断正确。先由 contract 算 expected values：


In [ ]:
state=0; threshold=4
print('cycle | input | before | candidate | spike | after')
for cycle,current in enumerate([1,1,1,1,2,2]):
    before=state; candidate=before+current; spike=candidate>=threshold
    state=0 if spike else candidate
    print(f'{cycle:5d} | {current:5d} | {before:6d} | {candidate:9d} | {int(spike):5d} | {state:5d}')


## 4. self-checking testbench

`tb/learning/tutorial_if_neuron_tb.sv` 在每个 edge 后比较 `membrane_v` / `spike`。不匹配就 `$fatal`；全部通过才打印 `PASS lesson08 tutorial_if_neuron`。


## 5. Run

若 Icarus Verilog 可用，下面实际 compile + simulation；否则明确显示未运行。


In [ ]:
from pathlib import Path
import shutil, subprocess, tempfile

def repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p/'pyproject.toml').exists() and (p/'lessons').exists(): return p
    raise FileNotFoundError('Run inside FPGA-FlyBrain')

root=repo_root(); iverilog=shutil.which('iverilog'); vvp=shutil.which('vvp')
if not (iverilog and vvp):
    print('Icarus Verilog not found; RTL simulation did not run.')
else:
    with tempfile.TemporaryDirectory() as td:
        td=Path(td); out=td/'l8.out'
        subprocess.run([iverilog,'-g2012','-o',str(out),str(root/'rtl/learning/tutorial_if_neuron.sv'),str(root/'tb/learning/tutorial_if_neuron_tb.sv')],check=True,cwd=td)
        r=subprocess.run([vvp,str(out)],check=True,text=True,capture_output=True,cwd=td)
        print(r.stdout.strip())
        print('A temporary tutorial_if_neuron.vcd waveform was generated during the run.')


## 6. waveform / VCD

testbench 用 `$dumpfile/$dumpvars` 记录波形；常见格式是 **Value Change Dump (VCD)**。看波形时要能回答：input 在哪个 edge 前稳定、spike 属于哪个 cycle、reset 后 state 从哪个 edge 可见。


## 7. 失败时先查哪层？

1. contract/oracle 是否正确；
2. testbench 是否在正确时间驱动/采样；
3. RTL combinational/register update 是否错误。

不要让 AI 随机改 RTL 直到变绿。


## 8. Try It / AI Task / Human Check

在临时副本把 `>=` 改成 `>`，先预测哪个 boundary case 会失败，再运行。恢复原代码。

让 AI 根据 failure message 提证据驱动的检查点，而非直接修补。

不用 AI 应能解释 simulation vs FPGA、testbench vs design、self-checking 与 waveform 各自作用。


## 9. Engineering Handoff / Project Trace

本课建立 RMD-005/RMD-005A 的验证习惯。正式 LIF RTL 将使用 Python fixed-point vectors 作直接 oracle；tutorial testbench 不替代它。

- Lesson: `LSN-008`
- Mapping: `RMD-005 / RMD-005A`
- Prepared level: `L3 RTL unit simulation`


## 10. Exit Ticket

你能读小型 RTL、区分 combinational/register、解释 self-checking testbench 与 waveform，并且不会把“simulation passed”说成“已经在 FPGA 上运行”。
